In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

# CIFAR-10 normalization statistics calculated from the training set
mean = [0.4914, 0.4822, 0.4465]
std = [0.2470, 0.2435, 0.2616]

normalized_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Load CIFAR-10 datasets
train_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=True,
    download=False,
    transform=normalized_transform
)

val_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=True,
    download=False,
    transform=normalized_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=normalized_transform
)

# Create stratified train/validation indices
all_indices = np.arange(len(train_dataset))

train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.1,
    random_state=42,
    stratify=train_dataset.targets
)

# Create train and validation subsets
train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(val_dataset, val_indices)

print("Training samples:", len(train_subset))
print("Validation samples:", len(val_subset))
print("Test samples:", len(test_dataset))

Training samples: 45000
Validation samples: 5000
Test samples: 10000


In [2]:
# Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_subset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_subset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0
)

# Inspect one training batch
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)

Batch image shape: torch.Size([128, 3, 32, 32])
Batch label shape: torch.Size([128])


In [3]:
import torch.nn as nn

class BaselineCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=3,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BaselineCNN()

print(model)

images, label = next(iter(train_loader))
outputs = model(images)

print("\nInput shape:", images.shape)
print("Output shape:", outputs.shape)

BaselineCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)

Input shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])


In [4]:
# Set random seed for reproducibility
torch.manual_seed(42)

# Select device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Initialize model
model = BaselineCNN().to(device)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

# Check loss on one batch
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

outputs = model(images)
loss = criterion(outputs, labels)

print("\nOutput shape:", outputs.shape)
print("Labels shape:", labels.shape)
print("Initial loss:", loss.item())

Device: cuda

Output shape: torch.Size([128, 10])
Labels shape: torch.Size([128])
Initial loss: 2.305391550064087


In [5]:
num_epochs = 10

train_losses = []
val_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(num_epochs):
    # Training
    model.train()

    running_train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss = running_train_loss / train_total
    train_accuracy = train_correct / train_total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)

    # Validation
    model.eval()

    running_val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = running_val_loss / val_total
    val_accuracy = val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f}"
    )

Epoch 01/10 | Train Loss: 1.5028 | Train Acc: 0.4585 | Val Loss: 1.1616 | Val Acc: 0.5844
Epoch 02/10 | Train Loss: 1.1395 | Train Acc: 0.5964 | Val Loss: 0.9737 | Val Acc: 0.6554
Epoch 03/10 | Train Loss: 0.9966 | Train Acc: 0.6459 | Val Loss: 0.9552 | Val Acc: 0.6598
Epoch 04/10 | Train Loss: 0.8988 | Train Acc: 0.6818 | Val Loss: 0.8561 | Val Acc: 0.6968
Epoch 05/10 | Train Loss: 0.8164 | Train Acc: 0.7110 | Val Loss: 0.8219 | Val Acc: 0.7050
Epoch 06/10 | Train Loss: 0.7488 | Train Acc: 0.7363 | Val Loss: 0.8252 | Val Acc: 0.7068
Epoch 07/10 | Train Loss: 0.6896 | Train Acc: 0.7563 | Val Loss: 0.7995 | Val Acc: 0.7178
Epoch 08/10 | Train Loss: 0.6339 | Train Acc: 0.7740 | Val Loss: 0.7998 | Val Acc: 0.7242
Epoch 09/10 | Train Loss: 0.5811 | Train Acc: 0.7900 | Val Loss: 0.8411 | Val Acc: 0.7122
Epoch 10/10 | Train Loss: 0.5408 | Train Acc: 0.8044 | Val Loss: 0.8028 | Val Acc: 0.7328
